In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score, recall_score, precision_score, roc_auc_score

import torch
import torch.nn as nn

# Modellenmeye hazır veri setinin okunması
df = pd.read_csv("preprocessed_common.csv") 


In [2]:
# Hedef değişken
target = "is_onview_arrest"

# özellik/hedef ayrımı
X = df.drop(columns=[target])
y = df[target]


In [3]:
# One-Hot Encoding
# Kategorik veriler sayısallaştırıldı
categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns

X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

print("One-hot sonrası boyut:", X.shape)


One-hot sonrası boyut: (47444, 8653)


In [4]:
# Train/Test Bölme
# Sınıf dengesini korumak için stratify=y kullandık
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [5]:
# VarianceThreshold ile düşük varyanslı özellikleri kaldırma
from sklearn.feature_selection import VarianceThreshold
vt = VarianceThreshold(threshold=0)
X_train = vt.fit_transform(X_train)
X_test = vt.transform(X_test)

# StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [6]:
# Feature Selection
from sklearn.feature_selection import SelectKBest, f_classif

actual_features = vt.get_feature_names_out() 

# ANOVA F-testi (SelectKBest) kullanılarak 
#hedef değişkenle istatistiksel olarak en anlamlı ilişkiye sahip olan en iyi K adet özellik seçildi
K = min(1000, len(actual_features))

selector = SelectKBest(score_func=f_classif, k=K)
X_train_fs = selector.fit_transform(X_train_scaled, y_train)
X_test_fs  = selector.transform(X_test_scaled)

selected_mask = selector.get_support()
selected_features = actual_features[selected_mask]

print(f"✔ En iyi {K} özellik seçildi.")
print("Seçilen ilk 10 özellik:", selected_features[:10])

✔ En iyi 1000 özellik seçildi.
Seçilen ilk 10 özellik: ['zipcode' 'arrest_year' 'arrest_month' 'arrest_day_of_week'
 'location_ 10TH ST / S SMITH RD  ' 'location_ 11TH ST / S MILL AVE  '
 'location_ 19TH ST / S ROOSEVELT ST  ' 'location_ 3RD ST / S MILL AVE  '
 'location_ 48TH ST / SR 143  ' 'location_ 48TH ST / W BASELINE RD  ']


In [7]:
# Sınıf dağılımı ve ağırlık hesabı
class_counts = y_train.value_counts()
print(class_counts)

pos_weight = class_counts[0] / class_counts[1]
print("Pozitif sınıf ağırlığı:", pos_weight)


is_onview_arrest
0    28805
1     9150
Name: count, dtype: int64
Pozitif sınıf ağırlığı: 3.148087431693989


In [8]:
# PyTorch modelinin çalışabilmesi için veriyi uygun formata dönüştürme
device = "cuda" if torch.cuda.is_available() else "cpu"

Xtr = torch.tensor(X_train_fs, dtype=torch.float32).to(device)
Xt = torch.tensor(X_test_fs, dtype=torch.float32).to(device)

ytr = torch.tensor(y_train.values, dtype=torch.float32).to(device)
yt = torch.tensor(y_test.values, dtype=torch.float32).to(device)


In [9]:
# Modelin tanımı
class AdvancedDNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze()


In [10]:
# Model eğitimi için denenecek hiperparametrelerin aralıkları tanımlandı
param_grid = {
    "hidden_dim": [64, 128, 256],
    "dropout": [0.2, 0.3, 0.4],
    "lr": [1e-3, 5e-4],
}


In [11]:
# Model eğitim fonksiyonu
def train_model(params, max_epochs=50, patience=5):
    model = AdvancedDNN(
        input_dim=Xtr.shape[1],
        hidden_dim=params["hidden_dim"],
        dropout=params["dropout"]
    ).to(device)

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(pos_weight).to(device)
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"])

    best_auc = 0
    patience_counter = 0

    for epoch in range(max_epochs):
        # --Train--
        model.train()
        optimizer.zero_grad()

        logits = model(Xtr)
        loss = criterion(logits, ytr)
        loss.backward()
        optimizer.step()

        # --Validation--
        model.eval()
        with torch.no_grad():
            probs = torch.sigmoid(model(Xt)).cpu().numpy()
            auc = roc_auc_score(y_test, probs)

        if auc > best_auc:
            best_auc = auc
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    return best_auc, model


In [12]:
# Random Search ile en iyi modelin seçilmesi
import random

results = []
best_model = None
best_score = 0

for _ in range(6):  
    params = {
        k: random.choice(v) for k, v in param_grid.items()
    }

    auc, model = train_model(params)

    results.append((params, auc))
    print(params, "AUC:", auc)

    if auc > best_score:
        best_score = auc
        best_model = model


{'hidden_dim': 64, 'dropout': 0.4, 'lr': 0.001} AUC: 0.8290043607968203
{'hidden_dim': 128, 'dropout': 0.4, 'lr': 0.001} AUC: 0.8326616750490171
{'hidden_dim': 128, 'dropout': 0.3, 'lr': 0.0005} AUC: 0.828031454207506
{'hidden_dim': 256, 'dropout': 0.3, 'lr': 0.0005} AUC: 0.8302993137608122
{'hidden_dim': 128, 'dropout': 0.3, 'lr': 0.0005} AUC: 0.8268479368153026
{'hidden_dim': 64, 'dropout': 0.2, 'lr': 0.0005} AUC: 0.8253036497941719


In [14]:
# Final model ile tahmin
best_model.eval()

with torch.no_grad():
    probs = torch.sigmoid(best_model(Xt)).cpu().numpy()
    preds = (probs > 0.6).astype(int)

# Performans değerlendirmesi
print("=== Deep Neural Network - DNN Classifier ===")
print("Accuracy:", accuracy_score(y_test, preds))
print("Precision:", precision_score(y_test, preds))
print("Recall:", recall_score(y_test, preds))
print("F1:", f1_score(y_test, preds))
print("ROC-AUC:", roc_auc_score(y_test, probs))

print("-" * 30)
print("Sınıflandırma Raporu:")
print(classification_report(y_test, preds, digits=4))

=== Deep Neural Network - DNN Classifier ===
Accuracy: 0.7949204341869534
Precision: 0.5702547247329499
Recall: 0.6066433566433567
F1: 0.5878864887759424
ROC-AUC: 0.8326616750490171
------------------------------
Sınıflandırma Raporu:
              precision    recall  f1-score   support

           0     0.8724    0.8547    0.8635      7201
           1     0.5703    0.6066    0.5879      2288

    accuracy                         0.7949      9489
   macro avg     0.7213    0.7307    0.7257      9489
weighted avg     0.7996    0.7949    0.7970      9489

